# zeroshot-preprocessing: Feature Overview

This notebook walks through the main capabilities of the `zspreprocessing` library — automatic, rule-based preprocessing pipeline selection for scikit-learn.

**Pipeline stages:**
1. **Profile** the dataset (single pass, no model fitting)
2. **Select** a scaler and feature reducer based on deterministic rules
3. **Build & fit** a sklearn-compatible pipeline

No hyperparameter search, no cross-validation.

In [1]:
import numpy as np
import scipy.sparse as sp
from sklearn.datasets import make_classification, make_regression

from zspreprocessing import (
    ZeroShotClassifierPreprocessor,
    ZeroShotRegressorPreprocessor,
    PreprocessorArtifact,
    inspect,
    select_scaler,
    select_reducer,
)

rng = np.random.default_rng(42)

---
## 1. Dataset Profiling with `inspect()`

`inspect(X, y)` scans your dataset once and returns a `PreprocessingProfile` — a dataclass holding ~17 statistics used to drive all downstream decisions.

In [2]:
# Typical dense classification dataset
X, y = make_classification(n_samples=500, n_features=40, n_informative=10,
                            n_redundant=10, random_state=42)

profile = inspect(X, y, task="classification")
print(profile)

PreprocessingProfile(
  n_samples=500, n_features=40, n_p_ratio=12.50
  sparsity=0.000, is_sparse_counts=False
  binary_feature_fraction=0.000
  median_feature_skewness=0.097, outlier_fraction=0.000
  near_zero_variance_fraction=0.000, median_abs_correlation=0.050
  feature_signal_p90=0.270
  task='classification'
  n_minority_class=249
)


In [3]:
# Key fields you can access directly
print(f"Samples / Features       : {profile.n_samples} / {profile.n_features}")
print(f"n/p ratio                : {profile.n_p_ratio:.2f}")
print(f"Sparsity                 : {profile.sparsity:.3f}")
print(f"Binary feature fraction  : {profile.binary_feature_fraction:.3f}")
print(f"Median feature skewness  : {profile.median_feature_skewness:.3f}")
print(f"Outlier fraction         : {profile.outlier_fraction:.3f}")
print(f"Median abs correlation   : {profile.median_abs_correlation:.3f}")
print(f"Near-zero variance frac  : {profile.near_zero_variance_fraction:.3f}")
print(f"Is sparse counts?        : {profile.is_sparse_counts}")
print(f"Task                     : {profile.task}")

Samples / Features       : 500 / 40
n/p ratio                : 12.50
Sparsity                 : 0.000
Binary feature fraction  : 0.000
Median feature skewness  : 0.097
Outlier fraction         : 0.000
Median abs correlation   : 0.050
Near-zero variance frac  : 0.000
Is sparse counts?        : False
Task                     : classification


---
## 2. Rule-Based Scaler Selection

`select_scaler(profile)` applies six ordered rules to pick one of four scalers:

| Rule | Condition | Scaler |
|------|-----------|--------|
| 1 | Sparse fingerprint-like data | `MaxAbsScaler` |
| 2 | ≥80% binary features | `MaxAbsScaler` |
| 3 | Sparsity > 50% | `MaxAbsScaler` |
| 4 | Outlier fraction > 30% | `RobustScaler` |
| 5 | Median skewness > 1.5 | `PowerTransformer` (Yeo-Johnson) |
| 6 | Default | `StandardScaler` |

In [4]:
def demo_scaler(X, y, label, task="classification"):
    p = inspect(X, y, task=task)
    s = select_scaler(p)
    r = select_reducer(p)
    print(f"{label:40s}  scaler={s:20s}  reducer={r}")

n = 300

# --- Standard: normal features, low sparsity, low skew, few outliers
X_std = rng.standard_normal((n, 30))
y_std = rng.integers(0, 2, n)
demo_scaler(X_std, y_std, "Normal features (standard)")

# --- Robust: heavy outliers
X_out = rng.standard_normal((n, 30))
mask = rng.random((n, 30)) < 0.1
X_out[mask] *= 50
demo_scaler(X_out, y_std, "Heavy outliers (>30%)")

# --- Power: highly skewed features
X_skew = rng.exponential(scale=1.0, size=(n, 30)) ** 3
demo_scaler(X_skew, y_std, "Highly skewed features")

# --- MaxAbs: sparse data
X_sparse = (rng.random((n, 100)) < 0.05).astype(float)
demo_scaler(X_sparse, y_std, "Sparse data (5% nonzero)")

# --- MaxAbs: binary fingerprint-like
X_fp = rng.integers(0, 2, (n, 100)).astype(float)
demo_scaler(X_fp, y_std, "Binary fingerprints")

Normal features (standard)                scaler=standard              reducer=variance_threshold
Heavy outliers (>30%)                     scaler=robust                reducer=variance_threshold
Highly skewed features                    scaler=robust                reducer=variance_threshold
Sparse data (5% nonzero)                  scaler=max_abs               reducer=correlation_filter
Binary fingerprints                       scaler=max_abs               reducer=correlation_filter


---
## 3. Rule-Based Reducer Selection

`select_reducer(profile)` picks between two strategies:

| Condition | Reducer |
|-----------|---------|
| ≤50 features **or** n/p ≥ 20 (well-determined) | `VarianceThreshold` (removes near-constant features) |
| High-dimensional underdetermined (n/p < 20 and p > 50) | `CorrelationFilter` (drops redundant correlated features) |

In [5]:
# Well-determined: many samples relative to features
X_wd = rng.standard_normal((2000, 50))
y_wd = rng.integers(0, 2, 2000)
p_wd = inspect(X_wd, y_wd, task="classification")
print(f"Well-determined  (n/p={p_wd.n_p_ratio:.1f}): {select_reducer(p_wd)}")

# Underdetermined: more features than usefully sampled
X_ud = rng.standard_normal((200, 500))
y_ud = rng.integers(0, 2, 200)
p_ud = inspect(X_ud, y_ud, task="classification")
print(f"Underdetermined  (n/p={p_ud.n_p_ratio:.2f}): {select_reducer(p_ud)}")

Well-determined  (n/p=40.0): variance_threshold
Underdetermined  (n/p=0.40): correlation_filter


---
## 4. High-Level API: `ZeroShotClassifierPreprocessor`

The top-level classes wrap profiling, selection, and pipeline construction into a single sklearn-compatible object.

In [6]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=800, n_features=80, n_informative=15,
                            n_redundant=30, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

pre = ZeroShotClassifierPreprocessor(verbose=True)
X_train_t = pre.fit_transform(X_train, y_train)
X_test_t  = pre.transform(X_test)

print(f"\nSelected scaler : {pre.scaler_name_}")
print(f"Selected reducer: {pre.reducer_name_}")
print(f"Features: {pre.n_features_in_} → {pre.n_features_out_}")
print(f"Train shape: {X_train_t.shape}")
print(f"Test  shape: {X_test_t.shape}")

────────────────────────────────────────────── ZeroShotPreprocessor ───────────────────────────────────────────────

Classification  n=640  |  p=80  |  n/p=8.0  |  skewness=0.09  |  outliers=0.00  |  corr=0.04

2026-04-07 13:28:11.469 | INFO     | zspreprocessing.utils.logging:info:50 - scaler=standard | reducer=correlation_filter


13:28:11 INFO     scaler=standard | reducer=correlation_filter


2026-04-07 13:28:11.475 | SUCCESS  | zspreprocessing.utils.logging:success:59 - scaler=standard | reducer=correlation_filter | 80 → 80 features


13:28:11 SUCCESS  scaler=standard | reducer=correlation_filter | 80 → 80 features



Selected scaler : standard
Selected reducer: correlation_filter
Features: 80 → 80
Train shape: (640, 80)
Test  shape: (160, 80)


In [7]:
# The underlying sklearn Pipeline is accessible
print(pre.pipeline_)

Pipeline(steps=[('imputer',
                 SimpleImputer(keep_empty_features=True, strategy='median')),
                ('vt0', VarianceThreshold(threshold=1e-06)),
                ('scaler', StandardScaler()),
                ('reducer',
                 Pipeline(steps=[('vt', VarianceThreshold(threshold=1e-06)),
                                 ('select', CorrelationFilter())]))])


---
## 5. Regression Support

`ZeroShotRegressorPreprocessor` works the same way for continuous targets. The profiler additionally captures `y_skewness` and `y_all_positive`.

In [8]:
X_r, y_r = make_regression(n_samples=600, n_features=60, n_informative=20, random_state=1)
X_r_train, X_r_test, y_r_train, _ = train_test_split(X_r, y_r, test_size=0.2)

pre_r = ZeroShotRegressorPreprocessor()
X_r_train_t = pre_r.fit_transform(X_r_train, y_r_train)
X_r_test_t  = pre_r.transform(X_r_test)

print(f"Scaler : {pre_r.scaler_name_}")
print(f"Reducer: {pre_r.reducer_name_}")
print(f"Features: {pre_r.n_features_in_} → {pre_r.n_features_out_}")
print(f"y_skewness   : {pre_r.profile_.y_skewness:.3f}")
print(f"y_all_positive: {pre_r.profile_.y_all_positive}")

2026-04-07 13:28:11.491 | INFO     | zspreprocessing.utils.logging:info:50 - scaler=standard | reducer=correlation_filter


2026-04-07 13:28:11.495 | SUCCESS  | zspreprocessing.utils.logging:success:59 - scaler=standard | reducer=correlation_filter | 60 → 60 features


Scaler : standard
Reducer: correlation_filter
Features: 60 → 60
y_skewness   : 0.097
y_all_positive: False


---
## 6. Sparse / Fingerprint Data (cheminformatics use case)

The library is specifically optimized for ECFP/Morgan fingerprints — binary sparse matrices common in drug discovery. It accepts both dense arrays and `scipy.sparse` matrices.

In [9]:
# Simulate ECFP-like fingerprints: sparse binary, integer values 0/1
n_mols, n_bits = 500, 1024
density = 0.04  # ~4% nonzero — typical for ECFP4

X_fp = sp.random(n_mols, n_bits, density=density, format="csr", random_state=42)
X_fp.data[:] = 1  # binarize
y_fp = rng.integers(0, 2, n_mols)

print(f"Input: {X_fp.shape}, sparsity={1 - X_fp.nnz / (n_mols * n_bits):.3f}")

pre_fp = ZeroShotClassifierPreprocessor()
X_fp_t = pre_fp.fit_transform(X_fp, y_fp)

print(f"Scaler : {pre_fp.scaler_name_}    (MaxAbsScaler preserves sparsity)")
print(f"Reducer: {pre_fp.reducer_name_}")
print(f"is_sparse_counts: {pre_fp.profile_.is_sparse_counts}")
print(f"Features: {pre_fp.n_features_in_} → {pre_fp.n_features_out_}")

2026-04-07 13:28:11.520 | INFO     | zspreprocessing.utils.logging:info:50 - scaler=max_abs | reducer=correlation_filter


Input: (500, 1024), sparsity=0.960


2026-04-07 13:28:11.673 | SUCCESS  | zspreprocessing.utils.logging:success:59 - scaler=max_abs | reducer=correlation_filter | 1024 → 1024 features


Scaler : max_abs    (MaxAbsScaler preserves sparsity)


Reducer: correlation_filter
is_sparse_counts: True
Features: 1024 → 1024


---
## 7. Missing Values and Constant Features

The pipeline always includes a `SimpleImputer(strategy="median")` and a `VarianceThreshold(1e-6)` as the first two steps — so you never need to handle these yourself.

In [10]:
X_miss = rng.standard_normal((300, 20)).astype(float)
# Add 15% missing values
mask = rng.random(X_miss.shape) < 0.15
X_miss[mask] = np.nan
# Add 2 constant columns
X_miss[:, 0] = 5.0
X_miss[:, 1] = 0.0
y_miss = rng.integers(0, 2, 300)

print(f"NaN count before: {np.isnan(X_miss).sum()}")
print(f"Input shape     : {X_miss.shape}")

pre_m = ZeroShotClassifierPreprocessor()
X_out = pre_m.fit_transform(X_miss, y_miss)

print(f"NaN count after : {np.isnan(X_out).sum()}")
print(f"Output shape    : {X_out.shape}  (2 constant cols removed)")

2026-04-07 13:28:11.694 | INFO     | zspreprocessing.utils.logging:info:50 - scaler=standard | reducer=variance_threshold


2026-04-07 13:28:11.699 | SUCCESS  | zspreprocessing.utils.logging:success:59 - scaler=standard | reducer=variance_threshold | 20 → 18 features


NaN count before: 794
Input shape     : (300, 20)


NaN count after : 0
Output shape    : (300, 18)  (2 constant cols removed)


---
## 8. ONNX Export

Fitted pipelines can be exported to ONNX for deployment.

In [11]:
import onnxruntime as rt
import tempfile, os

X_o, y_o = make_classification(n_samples=400, n_features=30, random_state=3)
pre_o = ZeroShotClassifierPreprocessor()
X_o_t = pre_o.fit_transform(X_o, y_o)

with tempfile.NamedTemporaryFile(suffix=".onnx", delete=False) as f:
    path = f.name

pre_o.to_onnx(path)

sess = rt.InferenceSession(path)
X_onnx = sess.run(None, {"float_input": X_o.astype("float32")})[0]

max_diff = np.abs(X_o_t - X_onnx).max()
print(f"Max absolute difference (sklearn vs ONNX): {max_diff:.2e}")
print(f"ONNX output shape: {X_onnx.shape}")
os.unlink(path)

2026-04-07 13:28:11.746 | INFO     | zspreprocessing.utils.logging:info:50 - scaler=standard | reducer=variance_threshold


2026-04-07 13:28:11.752 | SUCCESS  | zspreprocessing.utils.logging:success:59 - scaler=standard | reducer=variance_threshold | 30 → 30 features


Max absolute difference (sklearn vs ONNX): 4.60e-07
ONNX output shape: (400, 30)


---
## 8. Save, Load, and Run as an Artifact

`save()` writes two files to a directory: `preprocessor.onnx` and `preprocessor.json`.
`PreprocessorArtifact.load()` reads them back and exposes a `.run()` method for inference — no sklearn dependency required at serving time.

In [12]:
import tempfile, os

X_s, y_s = make_classification(n_samples=600, n_features=80, n_informative=15,
                                n_redundant=30, random_state=7)
X_train, X_test = X_s[:500], X_s[500:]
y_train         = y_s[:500]

# --- Fit and save ---
pre = ZeroShotClassifierPreprocessor()
X_train_t = pre.fit_transform(X_train, y_train)

save_dir = tempfile.mkdtemp(prefix="zspreprocessing_")
pre.save(save_dir, onnx=True)

print("Saved files:")
for f in sorted(os.listdir(save_dir)):
    size = os.path.getsize(os.path.join(save_dir, f))
    print(f"  {save_dir}/{f}  ({size:,} bytes)")

2026-04-07 13:28:11.918 | INFO     | zspreprocessing.utils.logging:info:50 - scaler=standard | reducer=correlation_filter


2026-04-07 13:28:11.923 | SUCCESS  | zspreprocessing.utils.logging:success:59 - scaler=standard | reducer=correlation_filter | 80 → 80 features


Saved files:
  /var/folders/yw/xtwk9yf95pn1k634kvjphtsh0000gn/T/zspreprocessing_rjsjxfal/preprocessor.json  (798 bytes)
  /var/folders/yw/xtwk9yf95pn1k634kvjphtsh0000gn/T/zspreprocessing_rjsjxfal/preprocessor.onnx  (1,932 bytes)


In [13]:
# --- Load as artifact and inspect metadata ---
artifact = PreprocessorArtifact.load(save_dir)
print(artifact)
print()

import json
with open(os.path.join(save_dir, "preprocessor.json")) as f:
    print("preprocessor.json:")
    print(json.dumps(json.load(f), indent=2))

PreprocessorArtifact(task='classification', scaler='standard', reducer='correlation_filter', features=80→80, backend='onnx')

preprocessor.json:
{
  "task": "classification",
  "scaler": "standard",
  "reducer": "correlation_filter",
  "n_features_in": 80,
  "n_features_out": 80,
  "kept_feature_indices": [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13,
    14,
    15,
    16,
    17,
    18,
    19,
    20,
    21,
    22,
    23,
    24,
    25,
    26,
    27,
    28,
    29,
    30,
    31,
    32,
    33,
    34,
    35,
    36,
    37,
    38,
    39,
    40,
    41,
    42,
    43,
    44,
    45,
    46,
    47,
    48,
    49,
    50,
    51,
    52,
    53,
    54,
    55,
    56,
    57,
    58,
    59,
    60,
    61,
    62,
    63,
    64,
    65,
    66,
    67,
    68,
    69,
    70,
    71,
    72,
    73,
    74,
    75,
    76,
    77,
    78,
    79
  ]
}


In [14]:
# --- Run artifact on test set and compare to sklearn pipeline ---
X_test_sklearn = pre.transform(X_test)
X_test_artifact = artifact.run(X_test)

max_diff = np.abs(X_test_sklearn - X_test_artifact).max()
print(f"sklearn  output shape : {X_test_sklearn.shape}")
print(f"artifact output shape : {X_test_artifact.shape}")
print(f"Max absolute difference: {max_diff:.2e}")
print(f"Results identical     : {max_diff < 1e-5}")

sklearn  output shape : (100, 80)
artifact output shape : (100, 80)
Max absolute difference: 4.38e-07
Results identical     : True
